# A PTP boundary-clock chain against a time-error budget

A telecom phase/time network carries time from a grandmaster through a chain of boundary clocks
(T-BC). Every hop adds time error: constant (link asymmetry) and dynamic (timestamp noise, the
servo, the oscillator). The end of the chain must stay within the network limit, typically
**1.1 µs** (ITU-T G.8271.1) for 1.5 µs applications. Each T-BC has its own noise-generation limits
by class (ITU-T G.8273.2).

This notebook simulates such chains (`ntpstats.ptpsim`) to answer two planning questions:

1. how does time error grow with the number of hops, and with link asymmetry;
2. what changes with the servo and with links through switches that do not support PTP.

These are models of linuxptp-style servos, not of a particular product; they show the trade-offs.

<sub>MIT licensed, part of [ntpstats](https://github.com/thiagodefreitas/NetworkTime). Runs in CI on every change.</sub>

In [ ]:
from pathlib import Path

import numpy as np

from ntpstats import api as nt
from ntpstats import ptpsim

# examples/data, whether the notebook runs from examples/notebooks or the repository root
DATA = next(p for p in (Path("../data"), Path("examples/data"), Path("../../examples/data")) if p.is_dir())

## Ten class-B-like boundary clocks

16 Sync and Delay_Req messages per second, hardware time stamps (4 ns rms per timestamp), the PI
servo, and up to ±10 ns of asymmetry per link (a few metres of fibre length difference).

In [ ]:
sc = ptpsim.ChainScenario(hops=10, duration=1800, asymmetry_spread=10e-9, seed=1)
res = ptpsim.simulate_chain(sc)
verdict = res.check(cls="B")
print(f"{'node':>4} {'asymmetry':>10} {'max|TE|':>9} {'|cTE|':>8} {'dTE_L MTIE':>11} {'hop adds |cTE|':>15}")
for n, h in zip(res.nodes, res.hops):
    print(f"{n['node']:>4} {nt.format_seconds(h['asymmetry']):>10} {nt.format_seconds(n['max_te']):>9} "
          f"{nt.format_seconds(n['cte']):>8} {nt.format_seconds(n['dte_l_mtie']):>11} {nt.format_seconds(h['cte']):>15}")
b = verdict["budget"]
print(f"end of chain {nt.format_seconds(b['value'])} vs {nt.format_seconds(b['limit'])}: "
      f"{'PASS' if b['passed'] else 'FAIL'}; every hop within class B: {verdict['hops_passed']}")

## How far can the chain go?

The same chain at 2 to 20 hops, with linuxptp's default servo gains; three seeds per length show
the spread.

In [ ]:
lengths = [2, 5, 10, 20]
ends = {}
for hops in lengths:
    ends[hops] = [ptpsim.simulate_chain(ptpsim.ChainScenario(hops=hops, duration=1200, asymmetry_spread=10e-9,
                                                              seed=s)).nodes[-1]["max_te"] for s in (1, 2, 3)]
    print(f"{hops:>3} hops: end max|TE| {', '.join(nt.format_seconds(v) for v in ends[hops])}")

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
if plt is not None:
    fig, ax = plt.subplots(figsize=(6, 3.5))
    for hops, v in ends.items():
        ax.plot([hops] * len(v), np.array(v) * 1e9, "o", color="C0")
    ax.axhline(1100, color="C3", ls="--", label="1.1 µs budget")
    ax.set(xlabel="boundary clocks", ylabel="end-of-chain max|TE| (ns)", yscale="log")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    plt.show()

Up to about ten hops the end of the chain stays within tens of nanoseconds. Beyond that the error
grows much faster than the number of hops: twenty hops reach around half a microsecond. The
cause is **gain peaking**. linuxptp's default PI gains are tuned for a single slave that follows
its master quickly: at 16 Sync/s the loop bandwidth is a few tenths of a hertz and it slightly
*amplifies* noise near that frequency. In a chain every hop amplifies the output of the previous
one, so the peaks multiply. This is why the T-BC requirements of G.8273.2 bound the loop bandwidth
and the gain peaking for clocks meant to be chained.

## Tuning the servo for a chain

A narrower, well-damped loop (about 0.05 Hz: `kp = 0.3`, `ki = 3.5e-4` per sample at 16 Sync/s)
follows the upstream more slowly but does not amplify its noise. It also takes longer to lock, so
both runs exclude the first 30 minutes.

In [ ]:
for label, opts in (("linuxptp defaults", {}), ("narrow loop (~0.05 Hz)", {"kp": 0.3, "ki": 3.5e-4})):
    r = ptpsim.simulate_chain(ptpsim.ChainScenario(hops=20, duration=3600, warmup=1800, asymmetry_spread=10e-9,
                                                    seed=1, servo_options=opts))
    end = r.nodes[-1]
    print(f"{label:>24}: 20 hops, end max|TE| {nt.format_seconds(end['max_te']):>9}, "
          f"dTE_L MTIE {nt.format_seconds(end['dte_l_mtie']):>9}, |cTE| {nt.format_seconds(end['cte']):>8}")

With a chain-friendly loop, twenty hops stay within a few tens of nanoseconds, and what is left is
mostly **asymmetry**: half of each link's asymmetry adds up, and two-way time transfer cannot
detect it. A single 1 µs fibre asymmetry costs 500 ns, so links must be calibrated, not just
measured. The command line runs the same study (`--kp`/`--ki`; the warm-up follows the servo):

```bash
ntpstats chain --hops 20 --kp 0.3 --ki 3.5e-4 --duration 1h --asymmetry-spread 10ns --class B
```

## Servo and network choices

The same 5-hop chain, with each servo, and with the boundary clocks connected through switches
without PTP support (queueing delay variation between them).

In [ ]:
from ntpstats.simulate import PathModel

cases = {
    "PI, on-path support": ptpsim.ChainScenario(hops=5, duration=1200, seed=4),
    "linreg, on-path support": ptpsim.ChainScenario(hops=5, duration=1200, seed=4, servo="linreg"),
    "PI, 1 µs queueing between BCs": ptpsim.ChainScenario(
        hops=5, duration=1200, seed=4, link=ptpsim.Link(pdv=PathModel(queue_mean=1e-6, load=0.3))),
}
for label, case in cases.items():
    end = ptpsim.simulate_chain(case).nodes[-1]
    print(f"{label:>32}: max|TE| {nt.format_seconds(end['max_te']):>9}, "
          f"dTE_L MTIE {nt.format_seconds(end['dte_l_mtie']):>9}, dTE_H {nt.format_seconds(end['dte_h_pp']):>9}")

The servo shapes how noise passes along the chain. Packet delay variation between boundary clocks
dominates everything else, which is why full on-path support (every switch a boundary clock) is
the norm for telecom phase/time.